# Module 1 project-exercise demo

**11 October 2026 edition.** Build two passports for a fictional document assistant.
The model record uses the same validated classifier as lecture Section 17. The system
record states the evidence, permitted actions, and acceptance rule. A second experiment
compares two fictional release-register snapshots.

This self-contained CPU exercise needs only the Python standard library: no downloads,
model API, or external packages. `Run all` completes the required path. Prepared answers
are checked against a narrow manual reference; no real LLM or general semantic judge is run.

In [ ]:
import json


## Demo 1. Two passports and an answer check

A configuration describes model structure. It does not determine which documents the
assistant can access or what makes an answer acceptable. Begin with the structural fields.
The functions below are shared with Section 17 and reject incompatible dimensions or
conflicting aliases. Configuration names identify fictional teaching variants, with no
claim about their answer quality.

In [ ]:
"""A minimal model passport from a validated config.json."""
import json

EXAMPLE_CONFIG_JSON = r"""
{
  "hidden_size": 2880,
  "num_hidden_layers": 24,
  "num_attention_heads": 64,
  "num_key_value_heads": 8,
  "num_local_experts": 32,
  "num_experts_per_tok": 4,
  "max_position_embeddings": 131072,
  "vocab_size": 201088,
  "rope_scaling": {"rope_type": "yarn", "factor": 32}
}
"""


def positive_int(config: dict, key: str) -> int:
    """Read one required positive integer field."""

    value = config.get(key)
    if isinstance(value, bool) or not isinstance(value, int) or value <= 0:
        raise ValueError(f"{key} must be a positive integer")
    return value


def one_of(config: dict, *keys: str):
    """Read one alias field without hiding conflicting values."""

    found = [(key, config[key]) for key in keys if config.get(key) is not None]
    if not found:
        return None
    if any(type(value) is not type(found[0][1]) or value != found[0][1]
           for _, value in found[1:]):
        raise ValueError(f"conflicting aliases: {[key for key, _ in found]}")
    return found[0][1]


def attention_kind(config: dict) -> str:
    """Classify MHA (Multi-Head Attention), MQA (Multi-Query Attention), GQA, or MLA from validated fields."""

    if "kv_lora_rank" in config:  # check the MLA signature first
        return f"MLA (kv_rank={positive_int(config, 'kv_lora_rank')})"
    heads = positive_int(config, "num_attention_heads")
    kv_heads = config.get("num_key_value_heads", heads)
    if isinstance(kv_heads, bool) or not isinstance(kv_heads, int) or kv_heads <= 0:
        raise ValueError("num_key_value_heads must be a positive integer")
    if kv_heads > heads or heads % kv_heads:
        raise ValueError("KV heads must divide Q heads")
    if kv_heads == heads:
        return f"MHA ({heads} heads)"
    if kv_heads == 1:
        return f"MQA ({heads} Q heads / 1 KV)"
    return f"GQA ({heads} Q heads / {kv_heads} KV)"


def moe_kind(config: dict) -> str:
    """Classify dense/MoE across common field aliases."""

    experts = one_of(config, "num_local_experts", "n_routed_experts", "num_experts")
    if experts is None:
        if any(config.get(k) is not None for k in ("num_experts_per_tok", "num_experts_per_token", "n_shared_experts", "num_shared_experts")):
            raise ValueError("expert settings require an expert count")
        return "dense"
    if isinstance(experts, bool) or not isinstance(experts, int) or experts <= 0:
        raise ValueError("expert count must be a positive integer")
    top_k = one_of(config, "num_experts_per_tok", "num_experts_per_token")
    if isinstance(top_k, bool) or not isinstance(top_k, int) or not 1 <= top_k <= experts:
        raise ValueError("top-k must be in [1, expert count]")
    shared = one_of(config, "n_shared_experts", "num_shared_experts")
    shared = 0 if shared is None else shared
    if isinstance(shared, bool) or not isinstance(shared, int) or shared < 0:
        raise ValueError("n_shared_experts must be a non-negative integer")
    return f"MoE: {experts} experts, top-{top_k}" + (f" + {shared} shared" if shared else "")


def passport(config: dict) -> dict:
    """Return a minimal structural passport without a network call."""

    return {
        "layers_x_hidden": (
            positive_int(config, "num_hidden_layers"),
            positive_int(config, "hidden_size"),
        ),
        "attention": attention_kind(config),
        "ffn": moe_kind(config),
        "context": positive_int(config, "max_position_embeddings"),
        "vocabulary": positive_int(config, "vocab_size"),
    }


In [ ]:
base = dict(hidden_size=64, num_hidden_layers=2, num_attention_heads=8,
            num_key_value_heads=2, max_position_embeddings=256, vocab_size=128)
configs = {
    "dense_gqa": dict(base),
    "toy_local_moe": dict(base, num_local_experts=4, num_experts_per_tok=2),
    "toy_mla_moe": dict(base, kv_lora_rank=16, n_routed_experts=8,
                        num_experts_per_tok=2, n_shared_experts=1),
}
for name, cfg in configs.items():
    print(name, passport(cfg))

assert "MoE" in passport(configs["toy_local_moe"])["ffn"]
assert passport(configs["toy_mla_moe"])["attention"].startswith("MLA")
for change in ({"num_key_value_heads": 3}, {"num_attention_heads": True},
               {"num_local_experts": 4, "num_experts_per_tok": 5},
               {"num_local_experts": 4, "num_experts": 8, "num_experts_per_tok": 2}):
    try:
        passport(dict(base, **change))
    except ValueError:
        pass
    else:
        raise AssertionError("invalid configuration accepted")


Now record the system. Inspect `model`, `latency`, and `cost`: prepared responses
are a teaching substitute, so the runtime of this checker cannot be reported as model
latency. The following cells reproduce the question, documents, and three cases from Section 12.

In [ ]:
system_passport = {'task': 'Answer the log-retention question with supporting evidence', 'documents': ['A', 'B'], 'model': 'Teaching substitute: prepared responses', 'permissions': 'Read-only; no network or writes', 'memory': 'Per question', 'acceptance': 'Correct period and manually labelled supporting quote', 'latency': 'Not measured for an LLM', 'cost': 'Not measured for an LLM'}
print(json.dumps(system_passport, ensure_ascii=False, indent=2))

### One question: inspect the checks

A document assistant helps a user answer a question and inspect the source. Our first
experiment uses two fictional instructions in the separate `document-assistant-v1`
fixture: one concerns experiment logs, the other temporary exports. The texts are
embedded here, so editing the lecture cannot change the experiment.

Read the question and both documents in the first cell. We then inspect three **prepared**
responses. No neural model or retrieval engine is run; the object of this experiment is
the acceptance rule. Each response has a `days` integer, a `document_id`, and a `quote`.
This structured representation makes the check visible without pretending to understand
arbitrary prose.

In [ ]:
DOCUMENTS = {'A': 'Keep an experiment log for 30 days after the experiment ends.', 'B': 'Delete a temporary export 7 days after it is created.'}
QUESTION = 'How many days must an experiment log be kept after the experiment ends?'
print(QUESTION)
for name, passage in DOCUMENTS.items():
    print(name, passage)


The errors differ. Seven days is wrong for an experiment log even with a real
citation to A. Thirty days is correct, but B does not support it. The checker therefore
returns separate fields for the answer, the presence of the quoted text, and agreement
with the manually labelled evidence.

`required_quote` is the complete reference passage for this question. This deliberately
narrow check may reject a valid paraphrase. It is not a general test of semantic support.

In [ ]:
def check_prepared_answer(answer, documents, reference):
    """Check a structured day count and an exact reference passage for one fixed question."""
    if type(answer.get("days")) is not int or answer["days"] < 0:
        raise ValueError("days must be a nonnegative integer")
    doc_id = answer.get("document_id")
    quote = answer.get("quote")
    if not isinstance(doc_id, str) or not isinstance(quote, str):
        raise ValueError("document_id and quote must be strings")
    answer_correct = answer["days"] == reference["days"]
    quote_exists = bool(quote) and doc_id in documents and quote in documents[doc_id]
    # The required passage is manually labelled for this question; it is not inferred.
    supported = (quote_exists and answer_correct
                 and doc_id == reference["document_id"]
                 and reference["required_quote"] in quote)
    return dict(answer_correct=answer_correct, quote_exists=quote_exists,
                supported=supported, accepted=answer_correct and supported)

In [ ]:
REFERENCE_ANSWER = {'days': 30, 'document_id': 'A', 'required_quote': 'Keep an experiment log for 30 days after the experiment ends.'}
prepared = [{'days': 30, 'document_id': 'A'}, {'days': 7, 'document_id': 'A'}, {'days': 30, 'document_id': 'B'}]
answers = [dict(row, quote=DOCUMENTS[row["document_id"]]) for row in prepared]
checks = [check_prepared_answer(row, DOCUMENTS, REFERENCE_ANSWER) for row in answers]
labels = ['Correct answer and source', 'Wrong retention period', 'Source about another object']
for label, result in zip(labels, checks):
    print(label, result)
assert checks == [{'answer_correct': True, 'quote_exists': True, 'supported': True, 'accepted': True}, {'answer_correct': False, 'quote_exists': True, 'supported': False, 'accepted': False}, {'answer_correct': True, 'quote_exists': True, 'supported': False, 'accepted': False}]


Only the first response is accepted. The second gets the period wrong; the third
uses evidence about a different object. Try replacing the first response's quote with
one letter or an invented sentence: it should no longer pass. Module 17 will build
retrieval and trace passage provenance; Module 19 will address evaluation more broadly.
For now we have made one accepted task concrete—the denominator in our later cost calculation.

## Demo 2. What changed between observations?

The second project tracks changes to a passport. This register is fictional so that new
releases cannot change the exercise. Dates in `previous` and `current` identify observations:
a later date alone is not a new model property. Compare fields by name, preserve unknown
input context as `None`, and track output limits separately. A real register also needs
a primary-source reference for each change.

In [ ]:
previous = {
    "model-a": {"access": "API", "context": 128_000, "max_output": 16_000, "date": "2026-08-06"},
    "model-b": {"access": "promised_weights", "context": 64_000, "max_output": 8_000, "date": "2026-08-06"},
    "model-d": {"access": "API", "context": 64_000, "max_output": 8_000, "date": "2026-08-06"},
}
current = {
    "model-a": {"access": "API", "context": 128_000, "max_output": 32_000, "date": "2026-10-11"},
    "model-b": {"access": "published_weights", "context": 64_000, "max_output": 8_000, "date": "2026-10-11"},
    "model-c": {"access": "restricted", "context": None, "max_output": 1_000_000, "date": "2026-10-11"},
    "model-d": {"access": "API", "context": 64_000, "max_output": 8_000, "date": "2026-10-11"},
}

def registry_changes(previous, current):
    changes = []
    for name in sorted(previous.keys() | current.keys()):
        if name not in previous:
            changes.append((name, "added", current[name]))
        elif name not in current:
            changes.append((name, "removed", previous[name]))
        else:
            delta = {key: (previous[name].get(key), current[name].get(key))
                     for key in sorted(previous[name].keys() | current[name].keys())
                     if key != "date" and previous[name].get(key) != current[name].get(key)}
            if delta:
                changes.append((name, "changed", delta))
    return changes

changes = registry_changes(previous, current)
for change in changes:
    print(change)
assert {name for name, *_ in changes} == {"model-a", "model-b", "model-c"}
assert current["model-c"]["context"] is None
assert registry_changes({"x":{"date":"old"}}, {"x":{"date":"new"}}) == []
assert registry_changes({"x":{}}, {}) == [("x", "removed", {})]


## Continuing to the full project

Start with one accessible artefact and fill both passports. Structural properties need
a primary source and an exact revision; measurements need data, settings, and acceptance
rules. Leave unknown fields unknown. You can then expand to five models and repeat the
observation several weeks later, keeping observation dates separate from release dates.

A real assistant must retrieve evidence, generate responses, retain those responses,
and evaluate them on held-out questions. Modules 17 and 19 provide the mechanisms.
Until that experiment is run, this demo establishes only the classifier, bounded
reference checks, and register comparison. It does not measure any LLM's answer quality.